# `payment_type` 03: related-feature analysis

            **Purpose:** identify predictors that represent the same concept, form a
            hierarchy, share a missingness process or plausibly interact with
            `payment_type`.

            ## Relationships selected in advance

            - `payment` — Payment is a fixed verbose relabelling of this field.
- `amount_tsh` — Payment arrangement provides context for a recorded tariff amount.
- `management` — Management arrangements may determine payment policy.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'payment_type'
feature_metadata = {'order': 30, 'name': 'payment_type', 'audit_type': 'category', 'role': 'candidate', 'disposition': 'retain as the canonical payment representation', 'finding': 'Seven fully covered levels include informative unknown and other states.', 'decision': 'Retain as nominal and do not reintroduce the removed payment alias.', 'risk': 'Payment patterns may proxy local administration and income.', 'sentinel_tokens': ['unknown'], 'related': [{'feature': 'payment', 'reason': 'Payment is a fixed verbose relabelling of this field.'}, {'feature': 'amount_tsh', 'reason': 'Payment arrangement provides context for a recorded tariff amount.'}, {'feature': 'management', 'reason': 'Management arrangements may determine payment policy.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for payment_type.


In [2]:
relationship_inventory = pd.DataFrame(feature_metadata["related"])
display(relationship_inventory)

relationship_evidence = related_feature_summary(
    training_features,
    feature,
    feature_metadata["audit_type"],
    feature_metadata["related"],
    feature_types,
)
display(relationship_evidence)


,feature,reason
0,payment,Payment is a fixed verbose relabelling of this...
1,amount_tsh,Payment arrangement provides context for a rec...
2,management,Management arrangements may determine payment ...


,primary,related,measure,association,complete rows,primary levels,related levels,forward modal purity (%),reverse modal purity (%),relationship rationale
0,payment_type,payment,bias-corrected Cramer's V,1.0000,59400,7,7,100.00,100.00,Payment is a fixed verbose relabelling of this...
1,payment_type,amount_tsh,correlation ratio (eta),0.2145,59400,7,98,NaN,NaN,Payment arrangement provides context for a rec...
2,payment_type,management,bias-corrected Cramer's V,0.2263,59400,7,12,68.73,47.21,Management arrangements may determine payment ...


In [3]:
payment_mapping = {
    "pay annually": "annually",
    "pay monthly": "monthly",
    "pay per bucket": "per bucket",
    "pay when scheme fails": "on failure",
    "never pay": "never pay",
    "other": "other",
    "unknown": "unknown",
}
expected_payment_type = training_features["payment"].map(payment_mapping)
payment_check = pd.DataFrame({
    "training rows": [len(training_features)],
    "mapped rows matching payment_type": [
        expected_payment_type.eq(training_features["payment_type"]).sum()
    ],
    "mismatches": [
        expected_payment_type.ne(training_features["payment_type"]).sum()
    ],
}, index=["payment -> payment_type"])
display(payment_check)


,training rows,mapped rows matching payment_type,mismatches
payment -> payment_type,59400,59400,0


## Discussion and modelling consequence

The relationships above were nominated before inspecting the pairwise
coefficients. A strong association can mean useful interaction, hierarchy,
shared collection behaviour or redundancy; it is not a reason to keep both
fields automatically.

For `payment_type`, carry the relationships into controlled ablations
and fit every learned grouping or encoding inside the training fold. The
current provisional disposition remains: **retain as the canonical payment representation**.
